In [19]:
import os
import glob
import pandas as pd
from pathlib import Path
from evaluate_llm import evaluate_llm
from evaluate_zap import evaluate_zap

In [20]:
# Get all CSV files from csv_vuln_list folder
csv_folder = "csv_vuln_list"
csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]
print(f"Found {len(csv_files)} CSV files: {csv_files}")


Found 4 CSV files: ['brokencrystals.csv', 'ginandjuice-shop.csv', 'juice_shop.csv', 'pixi.csv']


In [21]:
# Helper function to normalize CSV name to match ZAP/LLM naming conventions
def normalize_csv_name(csv_name):
    """Convert CSV name to match ZAP HTML and LLM JSON naming"""
    # Remove .csv extension
    name = csv_name.replace('.csv', '')
    # Convert underscore to hyphen for ZAP files
    zap_name = name.replace('_', '-')
    return name, zap_name

# Helper function to find all LLM reasoning files
def find_all_llm_files(csv_name):
    """Find all reasoning JSON files matching the pattern (all models and dates)"""
    # Try with original CSV name (with underscores)
    pattern1 = f"reasoning_result/*_{csv_name}_*.json"
    files1 = glob.glob(pattern1)

    # Try with hyphen version (for cases like juice_shop -> juice-shop)
    csv_name_hyphen = csv_name.replace('_', '-')
    pattern2 = f"reasoning_result/*_{csv_name_hyphen}_*.json"
    files2 = glob.glob(pattern2)

    # Combine and get unique files
    all_files = list(set(files1 + files2))

    # Sort by filename (which includes timestamp) - newest first
    all_files.sort(reverse=True)
    return all_files

# Helper function to extract date from LLM filename
def extract_date_from_filename(filename):
    """Extract and format date from filename YYYYMMDDHHMMSS_..."""
    if not filename:
        return None
    basename = os.path.basename(filename)
    # Extract timestamp (first 14 digits)
    if len(basename) >= 14 and basename[:14].isdigit():
        timestamp = basename[:14]
        # Format as YYYY-MM-DD HH:MM:SS
        date_str = f"{timestamp[:4]}-{timestamp[4:6]}-{timestamp[6:8]} {timestamp[8:10]}:{timestamp[10:12]}:{timestamp[12:14]}"
        return date_str
    return None

# Helper function to extract model name from LLM filename
def extract_model_from_filename(filename):
    """Extract model name from filename ..._<csv_name>_<model>.json"""
    if not filename:
        return None
    basename = os.path.basename(filename)
    # Remove .json extension
    basename = basename.replace('.json', '')
    # Split by underscore and get the last part (model name)
    parts = basename.split('_')
    if len(parts) >= 3:
        return parts[-1]  # Last part is the model name
    return None


In [22]:
# Evaluate all CSV files with all LLM files
results = []

for csv_file in csv_files:
    csv_path = os.path.join(csv_folder, csv_file)
    csv_name, zap_name = normalize_csv_name(csv_file)

    print(f"\n{'='*60}")
    print(f"Evaluating: {csv_name}")
    print(f"{'='*60}")

    # Find ZAP HTML file (only once per CSV)
    zap_html_path = os.path.join("zap_report", f"{zap_name}.html")
    if not os.path.exists(zap_html_path):
        print(f"Warning: ZAP file not found: {zap_html_path}")
        zap_result = None
    else:
        print(f"ZAP file: {zap_html_path}")
        try:
            zap_result = evaluate_zap(zap_html_path, csv_path)
            print(f"ZAP - TP: {zap_result['tp']}, FP: {zap_result['fp']}, FN: {zap_result['fn']}, "
                  f"Precision: {zap_result['precision']:.4f}, Recall: {zap_result['recall']:.4f}, "
                  f"F1: {zap_result['f1']:.4f}")
        except Exception as e:
            print(f"Error evaluating ZAP: {e}")
            zap_result = None

    # Find all LLM reasoning files
    llm_files = find_all_llm_files(csv_name)
    if not llm_files:
        print(f"Warning: No LLM files found for {csv_name}")
        # Still add result with None LLM
        results.append({
            'csv_name': csv_name,
            'zap_result': zap_result,
            'llm_result': None,
            'llm_file': None,
            'llm_date': None,
            'llm_model': None
        })
    else:
        print(f"Found {len(llm_files)} LLM file(s) for {csv_name}")
        # Evaluate each LLM file
        for llm_file in llm_files:
            print(f"  Evaluating LLM file: {os.path.basename(llm_file)}")
            try:
                llm_result = evaluate_llm(llm_file, csv_path)
                llm_date = extract_date_from_filename(llm_file)
                llm_model = extract_model_from_filename(llm_file)
                print(f"    Model: {llm_model}, Date: {llm_date}")
                print(f"    LLM - TP: {llm_result['tp']}, FP: {llm_result['fp']}, FN: {llm_result['fn']}, "
                      f"Precision: {llm_result['precision']:.4f}, Recall: {llm_result['recall']:.4f}, "
                      f"F1: {llm_result['f1']:.4f}")

                results.append({
                    'csv_name': csv_name,
                    'zap_result': zap_result,
                    'llm_result': llm_result,
                    'llm_file': llm_file,
                    'llm_date': llm_date,
                    'llm_model': llm_model
                })
            except Exception as e:
                print(f"    Error evaluating LLM file {llm_file}: {e}")
                results.append({
                    'csv_name': csv_name,
                    'zap_result': zap_result,
                    'llm_result': None,
                    'llm_file': llm_file,
                    'llm_date': extract_date_from_filename(llm_file),
                    'llm_model': extract_model_from_filename(llm_file)
                })



Evaluating: brokencrystals
ZAP file: zap_report\brokencrystals.html
ZAP - TP: 8, FP: 647, FN: 28, Precision: 0.0120, Recall: 0.2220, F1: 0.0230
Found 4 LLM file(s) for brokencrystals
  Evaluating LLM file: 20260108125746_brokencrystals_gpt-5.json
    Model: gpt-5, Date: 2026-01-08 12:57:46
    LLM - TP: 0, FP: 4, FN: 35, Precision: 0.0000, Recall: 0.0000, F1: 0.0000
  Evaluating LLM file: 20260108112746_brokencrystals_gpt-5-mini.json
    Model: gpt-5-mini, Date: 2026-01-08 11:27:46
    LLM - TP: 1, FP: 2, FN: 34, Precision: 0.3333, Recall: 0.0286, F1: 0.0526
  Evaluating LLM file: 20260104131914_brokencrystals_gpt-5.json
    Model: gpt-5, Date: 2026-01-04 13:19:14
    LLM - TP: 0, FP: 0, FN: 35, Precision: 0.0000, Recall: 0.0000, F1: 0.0000
  Evaluating LLM file: 20251214193345_brokencrystals_gpt-5-mini.json
    Model: gpt-5-mini, Date: 2025-12-14 19:33:45
    LLM - TP: 1, FP: 4, FN: 34, Precision: 0.2000, Recall: 0.0286, F1: 0.0500

Evaluating: ginandjuice-shop
ZAP file: zap_report\g

In [23]:
# Helper function to format app name
def format_app_name(csv_name):
    """Format CSV name to proper app name"""
    name_map = {
        'juice_shop': 'Juice Shop',
        'brokencrystals': 'BrokenCrystals',
        'ginandjuice-shop': 'Gin and Juice Shop',
        'pixi': 'Pixi'
    }
    return name_map.get(csv_name, csv_name.replace('_', ' ').title())

# Helper function to extract date part (YYYY-MM-DD) from datetime string
def extract_date_part(datetime_str):
    """Extract just the date part (YYYY-MM-DD) from datetime string (YYYY-MM-DD HH:MM:SS)"""
    if not datetime_str:
        return None
    # Split by space and take the first part (date)
    return datetime_str.split(' ')[0] if ' ' in datetime_str else datetime_str

def create_comparison_table(results, date=None, model=None):
    """
    Create a comparison table DataFrame from evaluation results.

    Args:
        results: List of evaluation results, each containing:
            - csv_name: Name of the CSV file
            - zap_result: ZAP evaluation result dict or None
            - llm_result: LLM evaluation result dict or None
            - llm_file: Path to LLM file or None
            - llm_date: Date string from LLM filename or None
            - llm_model: Model name from LLM filename or None
        date: Optional date string to filter LLM results (e.g., "2025-12-14" or "2025-12-14 19:21:31")
               Only the date part (YYYY-MM-DD) is used for comparison, time is ignored
        model: Optional model name to filter LLM results (e.g., "gpt-5-mini")

    Returns:
        pandas DataFrame with columns: App, Tool, Date, Model, TP, FP, FN, Precision, Recall, F1
    """
    # Extract date part from filter date if provided
    filter_date_part = extract_date_part(date) if date else None

    # Filter results based on date and model if provided
    filtered_results = []
    for result in results:
        if filter_date_part is not None or model is not None:
            # If filtering, only include results that match
            llm_date = result.get('llm_date', None)
            llm_model = result.get('llm_model', None)

            # Extract date part from llm_date for comparison
            llm_date_part = extract_date_part(llm_date) if llm_date else None

            # Check if this result matches the filter (compare date parts only)
            date_match = (filter_date_part is None) or (llm_date_part == filter_date_part)
            model_match = (model is None) or (llm_model == model)

            if date_match and model_match:
                filtered_results.append(result)
        else:
            # No filter, include all results
            filtered_results.append(result)

    # Get unique CSV names from filtered results
    csv_names = sorted(set(r['csv_name'] for r in filtered_results))

    # Group results by CSV name to get ZAP results (should be same for all)
    zap_results_by_csv = {}
    llm_results_by_csv = {}

    for result in filtered_results:
        csv_name = result['csv_name']

        # Store ZAP result (should be same for all results of same CSV)
        if csv_name not in zap_results_by_csv:
            zap_results_by_csv[csv_name] = result.get('zap_result', None)

        # Store LLM result if it matches the filter
        llm_date = result.get('llm_date', None)
        llm_model = result.get('llm_model', None)
        llm_result = result.get('llm_result', None)

        # Extract date parts for comparison
        llm_date_part = extract_date_part(llm_date) if llm_date else None

        # Only store if matches filter (or no filter) - compare date parts only
        if (filter_date_part is None or llm_date_part == filter_date_part) and (model is None or llm_model == model):
            if csv_name not in llm_results_by_csv or llm_result is not None:
                llm_results_by_csv[csv_name] = {
                    'result': llm_result,
                    'date': llm_date,
                    'model': llm_model
                }

    # Create comparison table
    comparison_data = []

    for csv_name in csv_names:
        app_name = format_app_name(csv_name)
        zap_result = zap_results_by_csv.get(csv_name, None)
        llm_info = llm_results_by_csv.get(csv_name, None)

        # Add ZAP row
        if zap_result:
            comparison_data.append({
                'App': app_name,
                'Tool': 'ZAP',
                'Date': None,
                'Model': None,
                'TP': zap_result['tp'],
                'FP': zap_result['fp'],
                'FN': zap_result['fn'],
                'Precision': zap_result['precision'],
                'Recall': zap_result['recall'],
                'F1': zap_result['f1']
            })
        else:
            comparison_data.append({
                'App': app_name,
                'Tool': 'ZAP',
                'Date': None,
                'Model': None,
                'TP': None,
                'FP': None,
                'FN': None,
                'Precision': None,
                'Recall': None,
                'F1': None
            })

        # Add LLM row
        if llm_info and llm_info['result']:
            comparison_data.append({
                'App': '',  # Empty for second row of same app
                'Tool': 'LLM',
                'Date': llm_info['date'],
                'Model': llm_info['model'],
                'TP': llm_info['result']['tp'],
                'FP': llm_info['result']['fp'],
                'FN': llm_info['result']['fn'],
                'Precision': llm_info['result']['precision'],
                'Recall': llm_info['result']['recall'],
                'F1': llm_info['result']['f1']
            })
        else:
            comparison_data.append({
                'App': '',  # Empty for second row of same app
                'Tool': 'LLM',
                'Date': llm_info['date'] if llm_info else None,
                'Model': llm_info['model'] if llm_info else None,
                'TP': None,
                'FP': None,
                'FN': None,
                'Precision': None,
                'Recall': None,
                'F1': None
            })

    # Create and return DataFrame
    return pd.DataFrame(comparison_data)

# Example: Get all unique model/date combinations from results
from collections import defaultdict
grouped_results = defaultdict(list)

for result in results:
    llm_model = result.get('llm_model', None)
    llm_date = result.get('llm_date', None)
    key = (llm_model, llm_date)
    grouped_results[key].append(result)

print("Available model/date combinations:")
for (model, date), group_results in sorted(grouped_results.items()):
    if model and date:
        print(f"  Model: {model}, Date: {date}")


Available model/date combinations:
  Model: gpt-5, Date: 2026-01-04 13:14:15
  Model: gpt-5, Date: 2026-01-04 13:16:30
  Model: gpt-5, Date: 2026-01-04 13:18:54
  Model: gpt-5, Date: 2026-01-04 13:19:14
  Model: gpt-5, Date: 2026-01-08 12:49:59
  Model: gpt-5, Date: 2026-01-08 12:51:44
  Model: gpt-5, Date: 2026-01-08 12:55:04
  Model: gpt-5, Date: 2026-01-08 12:57:46
  Model: gpt-5-mini, Date: 2025-12-14 19:21:31
  Model: gpt-5-mini, Date: 2025-12-14 19:24:59
  Model: gpt-5-mini, Date: 2025-12-14 19:29:33
  Model: gpt-5-mini, Date: 2025-12-14 19:33:45
  Model: gpt-5-mini, Date: 2026-01-08 11:08:05
  Model: gpt-5-mini, Date: 2026-01-08 11:11:02
  Model: gpt-5-mini, Date: 2026-01-08 11:15:21
  Model: gpt-5-mini, Date: 2026-01-08 11:27:46


In [24]:
# Example usage: Create comparison table with specific date and model
# The function returns a pandas DataFrame
# Note: Date filtering uses only the date part (YYYY-MM-DD), time is ignored

# Example 1: Get table for specific model and date (you can use just the date)
date = "2025-12-14"  # Can also use "2025-12-14 19:21:31" - time is ignored
model = "gpt-5-mini"
comparison_df = create_comparison_table(results, date=date, model=model)

# Display the pandas DataFrame (Jupyter will render it nicely)
comparison_df


,App,Tool,Date,Model,TP,FP,FN,Precision,Recall,F1
0,BrokenCrystals,ZAP,None,None,8,647,28,0.012,0.2220,0.0230
1,,LLM,2025-12-14 19:33:45,gpt-5-mini,1,4,34,0.200,0.0286,0.0500
2,Gin and Juice Shop,ZAP,None,None,2,1,16,0.667,0.1110,0.1900
3,,LLM,2025-12-14 19:24:59,gpt-5-mini,1,3,7,0.250,0.1250,0.1667
4,Juice Shop,ZAP,None,None,2,155,11,0.013,0.1540,0.0240
5,,LLM,2025-12-14 19:21:31,gpt-5-mini,2,6,17,0.250,0.1053,0.1481
6,Pixi,ZAP,None,None,4,457,1,0.009,0.8000,0.0170
7,,LLM,2025-12-14 19:29:33,gpt-5-mini,0,5,4,0.000,0.0000,0.0000


In [28]:
date = "2026-01-08"
model = "gpt-5-mini"
comparison_df = create_comparison_table(results, date=date, model=model)

# Display the pandas DataFrame (Jupyter will render it nicely)
comparison_df

,App,Tool,Date,Model,TP,FP,FN,Precision,Recall,F1
0,BrokenCrystals,ZAP,None,None,8,647,28,0.0120,0.2220,0.0230
1,,LLM,2026-01-08 11:27:46,gpt-5-mini,1,2,34,0.3333,0.0286,0.0526
2,Gin and Juice Shop,ZAP,None,None,2,1,16,0.6670,0.1110,0.1900
3,,LLM,2026-01-08 11:11:02,gpt-5-mini,1,3,7,0.2500,0.1250,0.1667
4,Juice Shop,ZAP,None,None,2,155,11,0.0130,0.1540,0.0240
5,,LLM,2026-01-08 11:08:05,gpt-5-mini,2,9,17,0.1818,0.1053,0.1333
6,Pixi,ZAP,None,None,4,457,1,0.0090,0.8000,0.0170
7,,LLM,2026-01-08 11:15:21,gpt-5-mini,0,9,4,0.0000,0.0000,0.0000


In [26]:
date = "2026-01-08"
model = "gpt-5"
comparison_df = create_comparison_table(results, date=date, model=model)

# Display the pandas DataFrame (Jupyter will render it nicely)
comparison_df

,App,Tool,Date,Model,TP,FP,FN,Precision,Recall,F1
0,BrokenCrystals,ZAP,None,None,8,647,28,0.0120,0.2220,0.0230
1,,LLM,2026-01-08 12:57:46,gpt-5,0,4,35,0.0000,0.0000,0.0000
2,Gin and Juice Shop,ZAP,None,None,2,1,16,0.6670,0.1110,0.1900
3,,LLM,2026-01-08 12:51:44,gpt-5,0,3,8,0.0000,0.0000,0.0000
4,Juice Shop,ZAP,None,None,2,155,11,0.0130,0.1540,0.0240
5,,LLM,2026-01-08 12:49:59,gpt-5,1,15,18,0.0625,0.0526,0.0571
6,Pixi,ZAP,None,None,4,457,1,0.0090,0.8000,0.0170
7,,LLM,2026-01-08 12:55:04,gpt-5,0,8,4,0.0000,0.0000,0.0000
